# Manipuler des données avec Pandas : statistiques et restructuration

Lino Galiana  
2026-07-20

<div class="badge-container"><div class="badge-text">Pour essayer les exemples présents dans ce tutoriel :</div><a href="https://github.com/linogaliana/python-datascientist-notebooks/blob/main/notebooks/manipulation/02_pandas_stats.ipynb" class="badge" target="_blank" rel="noopener"><img src="https://img.shields.io/static/v1?logo=github&label=&message=View%20on%20GitHub&color=181717" alt="View on GitHub"></a>
<a href="https://datalab.sspcloud.fr/launcher/ide/vscode-python?autoLaunch=true&name=«02_pandas_stats»&init.personalInit=«https%3A%2F%2Fraw.githubusercontent.com%2Flinogaliana%2Fpython-datascientist%2Fmain%2Fsspcloud%2Finit-vscode.sh»&init.personalInitArgs=«manipulation%2002_pandas_stats»" class="badge" target="_blank" rel="noopener"><img src="https://custom-icon-badges.demolab.com/badge/SSP%20Cloud-Lancer_avec_VSCode-blue?logo=vsc&logoColor=white" alt="Onyxia"></a>
<a href="https://datalab.sspcloud.fr/launcher/ide/jupyter-python?autoLaunch=true&name=«02_pandas_stats»&init.personalInit=«https%3A%2F%2Fraw.githubusercontent.com%2Flinogaliana%2Fpython-datascientist%2Fmain%2Fsspcloud%2Finit-jupyter.sh»&init.personalInitArgs=«manipulation%2002_pandas_stats»" class="badge" target="_blank" rel="noopener"><img src="https://img.shields.io/badge/SSP%20Cloud-Lancer_avec_Jupyter-orange?logo=Jupyter&logoColor=orange" alt="Onyxia"></a>
<a href="https://colab.research.google.com/github/linogaliana/python-datascientist-notebooks-colab//blob/main//notebooks/manipulation/02_pandas_stats.ipynb" class="badge" target="_blank" rel="noopener"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>
<a href="https://codespaces.new/linogaliana/python-datascientist-notebooks?quickstart=1&ref=main" class="badge" target="_blank" rel="noopener"><img src="https://github.com/codespaces/badge.svg" alt="Open in GitHub Codespaces"></a><br></div>

> **None**
>
> <div class="callout callout-style-default callout-note callout-titled">
> <div class="callout-header d-flex align-content-center">
> <div class="callout-icon-container">
> <i class="callout-icon"></i>
> </div>
> <div class="callout-title-container flex-fill">
> Note
> </div>
> </div>
> <div class="callout-body-container callout-body">
>
> Ceci est la version française 🇫🇷 de ce chapitre, pour voir la version anglaise rendez-vous sur <a href="https://pythonds.linogaliana.fr//home/runner/work/python-datascientist/python-datascientist/en/content/manipulation/02_pandas_stats.qmd">le site du cours</a>.
>
> </div>
> </div>


<div class="callout callout-style-default callout-tip callout-titled">
<div class="callout-header d-flex align-content-center">
<div class="callout-icon-container">
<i class="callout-icon"></i>
</div>
<div class="callout-title-container flex-fill">
Compétences à l’issue de ce chapitre
</div>
</div>
<div class="callout-body-container callout-body">

-   Construire des statistiques agrégées fines grâce aux méthodes de `Pandas` ;
-   Savoir restructurer ses données du format *long* au format *wide* (et inversement) ;
-   Créer des tableaux attractifs pour communiquer sur des résultats agrégés.

</div>
</div>

# 1. Introduction

Le [chapitre précédent](../../content/manipulation/02_pandas_joins.qmd) a montré comment enrichir un jeu de données en l’associant à une autre source grâce à une jointure. Nous disposons désormais de deux jeux de données mis en cohérence: les émissions de gaz à effet de serre de l’Ademe (`emissions`) et les données de cadrage Filosofi de l’Insee (`filosofi`).

Ce chapitre va exploiter cette association pour approfondir notre compréhension des données grâce à deux opérations complémentaires:

-   les statistiques descriptives par groupe ;
-   la restructuration de données entre formats *long* et *wide*.

Nous verrons enfin comment construire, à partir de ces statistiques, des tableaux de communication soignés grâce au *package* `great_tables`.

## 1.1 Données

Nous repartons des mêmes jeux de données que le [chapitre sur les jointures](../../content/manipulation/02_pandas_joins.qmd): les émissions communales de l’Ademe (`emissions`) et les données Filosofi de l’Insee (`filosofi`), enrichies l’une par l’autre. Le code ci-dessous, déjà commenté en détail dans le chapitre précédent, permet de reconstituer ces jeux de données:


In [ ]:
import requests
import pandas as pd

def get_melodi(dataset, dimension, code, value_name, time_period, geo_level="COM"):

  url = f"https://api.insee.fr/melodi/data/{dataset}"

  params = {
    "GEO": geo_level,
    dimension: code,
    "TIME_PERIOD": time_period,
    "maxResult": 40000,
  }

  response = requests.get(url, params=params, timeout=60)
  response.raise_for_status()
  observations = response.json()["observations"]

  return pd.DataFrame(
    {
      "CODGEO": obs["dimensions"]["GEO"].split("-")[-1],
      value_name: obs["measures"]["OBS_VALUE_NIVEAU"].get("value"),
    }
    for obs in observations
  )

niveau_vie = get_melodi("DS_FILOSOFI_CC", "FILOSOFI_MEASURE", "MED_SL", "NIVVIE_MEDIAN", 2023)
taux_pauvrete = get_melodi("DS_FILOSOFI_CC", "FILOSOFI_MEASURE", "PR_MD60", "TAUX_PAUVRETE", 2023)
population = get_melodi("DS_POPULATIONS_HISTORIQUES", "POPREF_MEASURE", "PMUN", "POPULATION", 2023)

url_cog_2023 = "https://www.insee.fr/fr/statistiques/fichier/6800675/v_commune_2023.csv"
url_backup = "https://minio.lab.sspcloud.fr/lgaliana/data/python-ENSAE/cog_2023.csv"

try:
  cog_2023 = pd.read_csv(url_cog_2023)
except requests.exceptions.Timeout:
  cog_2023 = pd.read_csv(url_backup)

noms_communes = (
  cog_2023
  .loc[cog_2023["TYPECOM"] == "COM", ["COM", "LIBELLE"]]
  .rename(columns={"COM": "CODGEO", "LIBELLE": "LIBGEO"})
)

filosofi = (
  niveau_vie
  .merge(taux_pauvrete, on="CODGEO", how="outer")
  .merge(population, on="CODGEO", how="outer")
  .merge(noms_communes, on="CODGEO", how="left")
  [["CODGEO", "LIBGEO", "POPULATION", "NIVVIE_MEDIAN", "TAUX_PAUVRETE"]]
)

filosofi["POPULATION"] = filosofi["POPULATION"].astype(int)
filosofi["dep"] = filosofi["CODGEO"].str[:2]

In [ ]:
import pandas as pd

url = "https://data.ademe.fr/data-fair/api/v1/datasets/igt-pouvoir-de-rechauffement-global/convert"
emissions = pd.read_csv(url)
emissions["dep"] = emissions["INSEE commune"].str[:2]

emissions.head(2)

Nous aurons également besoin des *packages* suivants

Pour obtenir des résultats reproductibles, on peut fixer la racine du générateur
pseudo-aléatoire.


In [ ]:
np.random.seed(123)

# 2. Statistiques descriptives par groupe

## 2.1 Principe

Nous avons vu, lors du chapitre d’introduction, comment obtenir
une statistique agrégée simplement grâce à `Pandas`.
Il est néanmoins commun d’avoir des données avec des strates
intermédiaires d’analyse pertinentes: des variables géographiques, l’appartenance à des groupes socio-démographiques liés à des caractéristiques renseignées, des indicatrices de période temporelle, etc.
Pour mieux comprendre la structure de ses données, les *data scientists* sont donc souvent amenés à construire des statistiques descriptives sur des sous-groupes présents dans les données.

Pour reprendre l’exemple sur les émissions, nous avions précédemment construit des statistiques d’émissions au niveau national. Mais qu’en est-il du profil d’émission des différents départements ? Pour répondre à cette question, il sera utile d’agréger nos données au niveau départemental. Ceci nous donnera une information différente du jeu de données initial (niveau communal) et du niveau le plus agrégé (niveau national).

En `SQL`, il est très simple de découper des données pour
effectuer des opérations sur des blocs cohérents et recollecter des résultats
dans la dimension appropriée.

La logique sous-jacente est celle du *split-apply-combine* qui est repris
par les langages de manipulation de données, auxquels `pandas`
[ne fait pas exception](https://pandas.pydata.org/pandas-docs/stable/user_guide/groupby.html).

L’image suivante, issue de
[ce site](https://unlhcc.github.io/r-novice-gapminder/16-plyr/),
représente bien la manière dont fonctionne l’approche
`split`-`apply`-`combine`:

<figure>
<img src="https://unlhcc.github.io/r-novice-gapminder/fig/12-plyr-fig1.png" alt="Split-apply-combine (Source: unlhcc.github.io)" />
<figcaption aria-hidden="true">Split-apply-combine (Source: <a href="https://unlhcc.github.io/r-novice-gapminder/16-plyr/">unlhcc.github.io</a>)</figcaption>
</figure>

En `Pandas`, on utilise `groupby` pour découper les données selon un ou
plusieurs axes (ce [tutoriel](https://realpython.com/pandas-groupby/) sur le sujet
est particulièrement utile).
L’ensemble des opérations d’agrégation (comptage, moyennes, etc.) que nous avions vues précédemment peut être mise en oeuvre par groupe.

Techniquement, cette opération consiste à créer une association
entre des labels (valeurs des variables de groupe) et des
observations. Utiliser la méthode `groupby` ne déclenche pas d’opérations avant la mise en oeuvre d’une statistique, cela créé seulement une relation formelle entre des observations et des regroupemens qui seront utilisés *a posteriori*:


In [ ]:
filosofi.groupby('dep').__class__

pandas.api.typing.DataFrameGroupBy

Tant qu’on n’appelle pas une action sur un `DataFrame` par groupe, du type
`head` ou `display`, `pandas` n’effectue aucune opération. On parle de
*lazy evaluation*. Par exemple, le résultat de `df.groupby('dep')` est
une transformation qui n’est pas encore évaluée :


In [ ]:
filosofi.groupby('dep')

## 2.2 Illustration 1: dénombrement par groupe

Pour illustrer l’application de ce principe à un comptage, on peut dénombrer le nombre de communes par département en 2023 (chaque année cette statistique change du fait des fusions de communes). Pour cela, il suffit de prendre le référentiel des communes françaises issu du code officiel géographique (COG) et dénombrer par département grâce à `count`:

Grâce à ce jeu de données, sans avoir recours aux statistiques par groupe, on peut déjà savoir combien on a, respectivement, de communes, départements et régions en France:


In [ ]:
communes = cog_2023.loc[cog_2023['TYPECOM']=="COM"]
communes.loc[:, ['COM', 'DEP', 'REG']].nunique()

COM    34945
DEP      101
REG       18
dtype: int64

Maintenant, intéressons nous aux départements ayant le plus de communes. Il s’agit de la même fonction de dénombrement où on joue, cette fois, sur le groupe à partir duquel est calculé la statistique.

Calculer cette statistique se fait de manière assez transparente lorsqu’on connaît le principe d’un calcul de statistiques avec `Pandas`:


In [ ]:
communes = cog_2023.loc[cog_2023['TYPECOM']=="COM"]
communes.groupby('DEP').agg({'COM': 'nunique'})

101 rows × 1 columns

En SQL, on utiliserait la requête suivante:

``` sql
SELECT dep, COUNT DISTINCT "COM" AS COM
FROM communes
GROUP BY dep
WHERE TYPECOM == 'COM';
```

La sortie est une `Serie` indexée. Ce n’est pas très pratique comme nous avons pu l’évoquer au cours du chapitre précédent. Il est plus pratique de transformer cet objet en `DataFrame` avec `reset_index`. Enfin, avec `sort_values`, on obtient la statistique désirée:


In [ ]:
(
    communes
    .groupby('DEP')
    .agg({'COM': 'nunique'})
    .reset_index()
    .sort_values('COM', ascending = False)
)

101 rows × 2 columns

## 2.3 Illustration 2: agrégats par groupe

Pour illustrer les agrégats par groupe nous pouvons prendre le jeu de données de l’Insee `filosofi` et sommer la variable `POPULATION`.

Pour calculer le total au niveau France entière nous pouvons faire de deux manières :


In [ ]:
filosofi['POPULATION'].sum()* 1e-6

np.float64(68.09428)

In [ ]:
filosofi.agg({"POPULATION": "sum"}).div(1e6)

POPULATION    68.09428
dtype: float64

où les résultats sont reportés en millions de personnes. La logique est identique lorsqu’on fait des statistiques par groupe, il s’agit seulement de remplacer `filosofi` par `filosofi.groupby('dep')` pour créer une version partitionnée par département de notre jeu de données:


In [ ]:
filosofi.groupby('dep')['POPULATION'].sum()

dep
01     679344
02     523342
03     333298
04     168054
05     143467
       ...   
92    1654712
93    1704316
94    1426929
95    1281653
97    1928465
Name: POPULATION, Length: 97, dtype: int64

In [ ]:
filosofi.groupby('dep').agg({"POPULATION": "sum"})

97 rows × 1 columns

La seconde approche est plus pratique car elle donne directement un `DataFrame` `Pandas` et non une série indexée sans nom. A partir de celle-ci, quelques manipulations basiques peuvent suffire pour avoir un tableau diffusables sur la démographie départementale. Néanmoins, celui-ci, serait quelques peu brut de décoffrage car nous ne possédons à l’heure actuelle que les numéros de département. Pour avoir le nom de départements, il faudrait utiliser une deuxième base de données et croiser les informations communes entre elles (en l’occurrence le numéro du département) : c’est le principe des jointures que nous avons mis en oeuvre dans le [chapitre précédent](../../content/manipulation/02_pandas_joins.qmd).

# 3. Exercice d’application

Cet exercice d’application s’appuie sur le jeu de données de l’Ademe nommé `emissions` précédemment.


<div class="callout callout-style-default callout-tip callout-titled">
<div class="callout-header d-flex align-content-center">
<div class="callout-icon-container">
<i class="callout-icon"></i>
</div>
<div class="callout-title-container flex-fill">
Exercice 1 : agrégations par groupe
</div>
</div>
<div class="callout-body-container callout-body">

1.  Calculer les émissions totales du secteur “Résidentiel” par département et rapporter la valeur au département le plus polluant dans le domaine. En tirer des intutitions sur la réalité que cette statistique reflète.

2.  Calculer, pour chaque département, les émissions totales de chaque secteur. Pour chaque département, calculer la proportion des émissions totales venant de chaque secteur.

<details>

<summary>

Indice pour cette question

</summary>

-   *“Grouper par”* = `groupby`
-   *“émissions totales”* = `agg({*** : "sum"})`

</details>

</div>
</div>

A la question 1, le résultat obtenu devrait être le suivant:


Ce classement reflète peut-être plus la démographie que le processus qu’on désire mesurer. Sans l’ajout d’une information annexe sur la population de chaque département pour contrôler ce facteur, on peut difficilement savoir s’il y a une différence structurelle de comportement entre les habitants du Nord (département 59) et ceux de la Moselle (département 57).

A l’issue de la question 2, prenons la part des émissions de l’agriculture et du secteur tertiaire dans les émissions départementales:


Ces résultats sont assez logiques ; les départements ruraux ont une part plus importante de leur émission issue de l’agriculture, les départements urbains ont plus d’émissions issues du secteur tertiaire, ce qui est lié à la densité plus importante de ces espaces.

Grâce à ces statistiques on progresse dans la connaissance de notre jeu de données et donc de la nature des émissions de C02 en France.
Les statistiques descriptives par groupe nous permettent de mieux saisir l’hétérogénéité spatiale de notre phénomène.

Cependant, on resterait limité dans notre capacité à interpréter ces statistiques sans recourir à de l’information annexe : un département pourrait sembler peu polluant simplement parce qu’il est peu peuplé. C’est exactement le type de limite que nous avons levée grâce à l’enrichissement de nos données par une jointure avec les données Filosofi (voir notamment le calcul d’une empreinte carbone par habitant dans le chapitre précédent).

# 4. Restructurer les données

## 4.1 Principe

Quand on a plusieurs informations pour un même individu ou groupe, on
retrouve généralement deux types de structure de données :

-   format **wide** : les données comportent des observations répétées, pour un même individu (ou groupe), dans des colonnes différentes
-   format **long** : les données comportent des observations répétées, pour un même individu, dans des lignes différentes avec une colonne permettant de distinguer les niveaux d’observations

Un exemple de la distinction entre les deux peut être pris à l’ouvrage de référence d’Hadley Wickham, [*R for Data Science*](https://r4ds.hadley.nz/):

<figure>
<img src="https://d33wubrfki0l68.cloudfront.net/3aea19108d39606bbe49981acda07696c0c7fcd8/2de65/images/tidy-9.png" alt="Données long et wide (Source: R for Data Science)" />
<figcaption aria-hidden="true">Données <em>long</em> et <em>wide</em> (Source: <a href="https://r4ds.hadley.nz/"><em>R for Data Science</em></a>)</figcaption>
</figure>

L’aide mémoire suivante aidera à se rappeler les fonctions à appliquer si besoin :

![](https://minio.lab.sspcloud.fr/lgaliana/generative-art/pythonds/reshape.png)

Le fait de passer d’un format *wide* au format *long* (ou vice-versa)
peut être extrêmement pratique car certaines fonctions sont plus adéquates sur une forme de données ou sur l’autre.

En règle générale, avec `Python` comme avec `R`, **les formats *long* sont souvent préférables**.
Les formats *wide* sont plutôt pensés pour des tableurs comme `Excel` ou on dispose d’un nombre réduit
de lignes à partir duquel faire des tableaux croisés dynamiques.

## 4.2 Exercice d’application

Les données de l’ADEME, et celles de l’Insee également, sont au format
*wide*.
Le prochain exercice illustre l’intérêt de faire la conversion *long* $\to$ *wide*
avant de faire un graphique avec la méthode `plot` vue au chapitre d’introduction


<div class="callout callout-style-default callout-tip callout-titled">
<div class="callout-header d-flex align-content-center">
<div class="callout-icon-container">
<i class="callout-icon"></i>
</div>
<div class="callout-title-container flex-fill">
Exercice 2: Restructurer les données : wide to long
</div>
</div>
<div class="callout-body-container callout-body">

1.  Créer une copie des données de l’`ADEME` en faisant `df_wide = emissions_wide.copy()`

2.  Restructurer les données au format *long* pour avoir des données d’émissions par secteur en gardant comme niveau d’analyse la commune (attention aux autres variables identifiantes).

3.  Faire la somme par secteur et représenter graphiquement

4.  Garder, pour chaque département, le secteur le plus polluant

</div>
</div>

# 5. Formatter des tableaux de statistiques descriptives

Un *dataframe* `Pandas`
est automatiquement mis en forme lorsqu’il est visualisé depuis un *notebook* sous forme de table HTML à la mise en forme minimaliste.
Cette mise en forme est pratique pour voir
les données, une tâche indispensable pour les *data scientists*
mais ne permet pas d’aller vraiment au-delà.

Dans une phase
exploratoire, il peut être pratique d’avoir un tableau
un peu plus complet, intégrant notamment des visualisations
minimalistes, pour mieux connaître ses données. Dans la phase
finale d’un projet, lorsqu’on communique sur un projet, il
est avantageux de disposer d’une visualisation attrative.
Pour ces deux besoins, les sorties des *notebooks* sont
une réponse peu satisfaisante, en plus de nécessiter
le *medium* du *notebook* qui peut en rebuter certains.

Heureusement, le tout jeune *package* [`great_tables`](https://posit-dev.github.io/great-tables/get-started/) permet, simplement, de manière programmatique, la création de tableaux
qui n’ont rien à envier à des productions manuelles fastidieuses faites dans `Excel`
et difficilement répliquables. Ce *package* est un portage en `Python` du *package* [`GT`](https://gt.rstudio.com/).
`great_tables` construit des tableaux
*html* ce qui offre une grande richesse dans la mise en forme et permet une excellente intégration avec [`Quarto`](https://quarto.org/), l’outil de publication reproductible développé par
L’exercice suivant proposera de construire un tableau avec
ce *package*, pas à pas. Il nécessite d’installer, au préalable, le *package* `great_tables`:


In [ ]:
!pip install great_tables --quiet

Afin de se concentrer sur la construction du tableau,
les préparations de données à faire en amont sont données
directement. Nous allons repartir du jeu de données `emissions_merged`, construit au chapitre précédent lors du calcul de l’empreinte carbone par habitant, qui a l’aspect suivant:

Pour être sûr d’être en mesure d’effectuer le prochain exercice, voici le dataframe nécessaire pour celui-ci


In [ ]:
emissions_totales_commune = emissions.sum(axis = 1, numeric_only = True)

emissions_merged = (
    emissions.assign(emissions = emissions_totales_commune)
    .reset_index()
    .merge(filosofi, left_on = "INSEE commune", right_on = "CODGEO")
)
emissions_merged = emissions_merged.loc[emissions_merged['POPULATION'] > 0]
emissions_merged['empreinte'] = emissions_merged['emissions']/emissions_merged['POPULATION']
emissions_merged['empreinte'] = emissions_merged['empreinte'].astype(float)

In [ ]:
emissions_table = (
    emissions_merged
    .rename(columns={"dep_y": "dep", "POPULATION": "population", "NIVVIE_MEDIAN": "revenu"})
    .groupby("dep")
    .agg({"empreinte": "sum", "revenu": "median", "population": "sum"}) #pas vraiment le revenu médian
    .reset_index()
    .sort_values(by = "empreinte")
)

Dans ce tableau nous allons intégrer des barres horizontales, à la manière des exemples présentés [ici](https://posit-dev.github.io/great-tables/examples/). Cela se fait en incluant directement le code *html* dans la colonne du *DataFrame*


In [ ]:
def create_bar(prop_fill: float, max_width: int, height: int, color: str = "green") -> str:
    """Create divs to represent prop_fill as a bar."""
    width = round(max_width * prop_fill, 2)
    px_width = f"{width}px"
    return f"""\
    <div style="width: {max_width}px; background-color: lightgrey;">\
        <div style="height:{height}px;width:{px_width};background-color:{color};"></div>\
    </div>\
    """

colors = {'empreinte': "green", 'revenu': "red", 'population': "blue"}

for variable in ['empreinte', 'revenu', 'population']:
    emissions_table[f'raw_perc_{variable}'] = emissions_table[variable]/emissions_table[variable].max()
    emissions_table[f'bar_{variable}'] = emissions_table[f'raw_perc_{variable}'].map(
        lambda x: create_bar(x, max_width=75, height=20, color = colors[variable])
    )

Nous ne gardons que les 5 plus petites empreintes carbone, et les cinq plus importantes.


In [ ]:
emissions_min = emissions_table.head(5).assign(grp = "5 départements les moins pollueurs").reset_index(drop=True)
emissions_max = emissions_table.tail(5).assign(grp = "5 départements les plus pollueurs").reset_index(drop=True)

emissions_table = pd.concat([
    emissions_min,
    emissions_max
])

Enfin, pour pouvoir utiliser quelques fonctions pratiques pour sélectionner des colonnes à partir de motifs, nous allons convertir les données au format [`Polars`](https://pola.rs/)


In [ ]:
import polars as pl
emissions_table = pl.from_pandas(emissions_table)


<div class="callout callout-style-default callout-tip callout-titled">
<div class="callout-header d-flex align-content-center">
<div class="callout-icon-container">
<i class="callout-icon"></i>
</div>
<div class="callout-title-container flex-fill">
Exercice 5: Un beau tableau de statistiques descriptives (exercice libre)
</div>
</div>
<div class="callout-body-container callout-body">

En prenant comme base ce tableau

``` python
GT(emissions_table, groupname_col="grp", rowname_col="dep")
```

construire un tableau dans le style de celui ci-dessous

</div>
</div>


In [ ]:
# Start from here
from great_tables import GT
GT(emissions_table, groupname_col="grp", rowname_col="dep")

Le tableau à obtenir:


Grâce à celui-ci, on peut déjà comprendre que notre définition
de l’empreinte carbone est certainement défaillante. Il apparaît
peu plausible que les habitants du 77 aient une empreinte 500 fois
supérieure à celle de Paris intra-muros. La raison principale ?
On n’est pas sur un concept d’émissions à la consommation mais à la
production, ce qui pénalise les espaces industriels ou les espaces
avec des aéroports…

Pour aller plus loin sur la construction de tableaux
avec `great_tables`, vous pouvez répliquer
cet [exercice](https://rgeo.linogaliana.fr/exercises/eval.html)
de production de tableaux électoraux
que j’ai proposé pour un cours de `R` avec `gt`, l’équivalent
de `great_tables` pour `R`.

Nous avons désormais fait le tour des principales fonctionnalités de `Pandas` pour manipuler et restructurer des données. Le [dernier chapitre](../../content/manipulation/02_pandas_beyond.qmd) de cette partie propose de prendre un peu de recul sur les limites de la syntaxe `Pandas` et de découvrir les écosystèmes alternatifs qui existent pour aller au-delà.
